# Bam-Dan Deep Analysis

This notebook implements the full analysis spec in one place.
It saves charts to `private_analysis/charts/`, tables to `private_analysis/tables/`,
and a narrative summary to `private_analysis/findings.md`.

In [14]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.use('Agg')

# Create output directories before any save calls run later in the notebook.
os.makedirs('private_analysis/charts', exist_ok=True)
os.makedirs('private_analysis/tables', exist_ok=True)

# Load the local parquet export created by fetch_data.py.
df = pd.read_parquet('data/responses.parquet')
print(f'Loaded {len(df)} responses')

# Drop duplicate demographic question columns because flat fields already contain them.
demographic_duplicate_cols = ['q_101', 'q_102', 'q_103', 'q_104', 'q_105']
df = df.drop(columns=demographic_duplicate_cols, errors='ignore')

# Keep only the 22 substantive survey questions for analysis.
question_cols = [f'q_{i}' for i in range(1, 23) if f'q_{i}' in df.columns]

# Convert question responses to numeric so comparisons and means behave consistently.
for question_column in question_cols:
    df[question_column] = pd.to_numeric(df[question_column], errors='coerce')

# Parse the timestamp once so later tasks can reuse the cleaned date fields.
df['createdAt_dt'] = pd.to_datetime(df['createdAt'], utc=True)
df['response_date'] = df['createdAt_dt'].dt.date

print(f'Question columns available: {question_cols}')
print(f'Gender breakdown: {df.gender.value_counts().to_dict()}')


Loaded 17259 responses
Question columns available: ['q_1', 'q_2', 'q_3', 'q_4', 'q_5', 'q_6', 'q_7', 'q_8', 'q_9', 'q_10', 'q_11', 'q_12', 'q_13', 'q_14', 'q_15', 'q_16', 'q_17', 'q_18', 'q_19', 'q_20', 'q_21', 'q_22']
Gender breakdown: {'male': 15313, 'female': 1781, 'other': 165}


In [15]:
# Colour palette from the spec. Use these values consistently across charts.
COLORS = {
    'primary':  '#2C3E7A',
    'red':      '#E8442A',
    'amber':    '#F5A623',
    'green':    '#27AE60',
    'purple':   '#8E44AD',
    'blue':     '#2980B9',
    'light_bg': '#F0F4FF',
    'text':     '#1A1A2E',
    'muted':    '#5A5A7A',
}

# Political label colours from the spec.
LABEL_COLORS = {
    'Leftist':         '#C0392B',
    'Center Left':     '#E67E22',
    'Centrist':        '#F39C12',
    'Center Right':    '#2980B9',
    'Rightwing':       '#1A5276',
    'Religious Left':  '#8E44AD',
    'Religious Right': '#6C3483',
}

# Multi-group categorical palette from the spec.
CAT_COLORS = [
    '#2C3E7A', '#E8442A', '#F5A623', '#27AE60',
    '#8E44AD', '#2980B9', '#E67E22', '#16A085'
]

# English question text from the spec. Do not rename or shorten these values.
QUESTION_TEXT = {
    1:  'State should control major industries & businesses',
    2:  'Business should be open to all — no govt interference',
    3:  'Tax the rich more to help the poor',
    4:  'Govt should not interfere in how businesses operate',
    5:  'Worker unions should be stronger',
    6:  'Foreign companies should get special incentives to invest',
    7:  'Education & healthcare should be completely free for all',
    8:  'Everyone has full rights over their own property',
    9:  'Laws should follow religious rules',
    10: 'Religion is personal — state should treat all equally',
    11: 'Traditional family & social norms matter more than modernity',
    12: 'Society should not restrict what people wear or how they behave',
    13: 'Religious education should be compulsory in schools',
    14: 'Freedom of speech — even if it hurts someone',
    15: 'Our culture must be protected from foreign influence',
    16: 'Men and women should have equal property rights',
    17: 'Strict rules & fines for factories to protect environment',
    18: 'Some environmental damage is acceptable for economic growth',
    19: 'State should equally support all religious festivals',
    20: 'Strictly ban anything that hurts religious sentiments',
    21: 'Indigenous/ethnic minorities deserve special rights & protection',
    22: 'Same rules for everyone — no special treatment for any group',
}

# Question categories from the spec.
QUESTION_CATEGORY = {
    1: 'economic', 2: 'economic', 3: 'economic', 4: 'economic',
    5: 'economic', 6: 'economic', 7: 'economic', 8: 'economic',
    9: 'social', 10: 'social', 11: 'social', 12: 'social',
    13: 'social', 14: 'social', 15: 'social', 16: 'social',
    17: 'economic', 18: 'economic', 19: 'social', 20: 'social',
    21: 'social', 22: 'social',
}

# Fixed display orders keep tables and charts easy to compare across tasks.
RESULT_LABEL_ORDER = [
    'Leftist', 'Center Left', 'Centrist', 'Center Right',
    'Rightwing', 'Religious Left', 'Religious Right'
]
AGE_ORDER = ['18-24', '25-34', '35-44', '45-54', '55+']
EDUCATION_ORDER = ['below_ssc', 'ssc_hsc', 'bachelor_masters', 'phd']
OCCUPATION_ORDER = ['student', 'service', 'business', 'other', 'homemaker']
SOCIAL_QUESTION_IDS = [9, 10, 11, 12, 14, 16, 20]
TOLERANCE_QUESTION_IDS = [12, 14, 16, 19, 21, 20]
ECONOMIC_OCCUPATION_QUESTION_IDS = [1, 3, 7, 8]

# Friendly English labels keep chart legends readable and non-technical.
EDUCATION_LABELS = {
    'below_ssc': 'Below SSC',
    'ssc_hsc': 'SSC/HSC',
    'bachelor_masters': 'Bachelor/Masters',
    'phd': 'PhD',
}
OCCUPATION_LABELS = {
    'student': 'Student',
    'service': 'Service/Employment',
    'business': 'Business',
    'other': 'Other',
    'homemaker': 'Homemaker',
}

# Sample-wide summary values are reused in multiple tasks and in findings.md.
total_count = len(df)
male_count = int(df['gender'].eq('male').sum())
female_count = int(df['gender'].eq('female').sum())
student_count = int(df['occupation'].eq('student').sum())
male_pct = male_count / total_count * 100
student_pct = student_count / total_count * 100
sample_start_date = pd.to_datetime(df['response_date']).min()
sample_end_date = pd.to_datetime(df['response_date']).max()
sample_year = int(sample_start_date.year)

# Set a consistent seaborn theme once so later plotting code stays cleaner.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = COLORS['light_bg']
plt.rcParams['axes.edgecolor'] = COLORS['muted']
plt.rcParams['axes.labelcolor'] = COLORS['text']
plt.rcParams['axes.titlecolor'] = COLORS['text']
plt.rcParams['xtick.color'] = COLORS['text']
plt.rcParams['ytick.color'] = COLORS['text']
plt.rcParams['text.color'] = COLORS['text']

def style_axes(plot_axes):
    """Apply the shared visual style rules to one axes object."""
    plot_axes.set_facecolor(COLORS['light_bg'])
    plot_axes.spines['top'].set_visible(False)
    plot_axes.spines['right'].set_visible(False)
    plot_axes.spines['left'].set_color(COLORS['muted'])
    plot_axes.spines['bottom'].set_color(COLORS['muted'])
    plot_axes.tick_params(colors=COLORS['text'])

def add_sample_caveat(plot_figure):
    """Add the required sample caveat subtitle to a figure."""
    plot_figure.text(
        0.5,
        -0.02,
        'Self-selected online sample | n=17,259 | April 2026 | bam-dan.com',
        ha='center',
        fontsize=9,
        color=COLORS['muted'],
        style='italic'
    )

def save_chart(plot_figure, file_name):
    """Save a chart to the required charts folder with the required export settings."""
    chart_path = os.path.join('private_analysis', 'charts', file_name)
    plot_figure.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')

def save_table(table_df, file_name):
    """Save a table to the required tables folder as CSV."""
    table_path = os.path.join('private_analysis', 'tables', file_name)
    table_df.to_csv(table_path, index=False)

def get_question_id(question_column):
    """Turn a column name like q_9 into the integer 9."""
    return int(question_column.split('_')[1])

def get_short_question_label(question_id):
    """Truncate question text to eight words for cramped axes labels."""
    question_words = QUESTION_TEXT[question_id].split()
    if len(question_words) <= 8:
        return ' '.join(question_words)
    return ' '.join(question_words[:8]) + '...'

def format_date_label(date_value):
    """Format dates without leading zeroes so labels match the spec style."""
    timestamp_value = pd.Timestamp(date_value)
    return f"{timestamp_value.strftime('%b')} {timestamp_value.day}"

def safe_ttest(group_a_series, group_b_series):
    """Run a Welch t-test and return NaN when a test is not possible."""
    clean_group_a = group_a_series.dropna()
    clean_group_b = group_b_series.dropna()
    if len(clean_group_a) < 2 or len(clean_group_b) < 2:
        return np.nan
    _, p_value = stats.ttest_ind(clean_group_a, clean_group_b, equal_var=False)
    return p_value

def add_count_and_pct_labels(plot_axes, counts, percentages):
    """Write count and percent labels at the end of horizontal bars."""
    for bar_index, count_value in enumerate(counts):
        percentage_value = percentages[bar_index]
        plot_axes.text(
            count_value + total_count * 0.01,
            bar_index,
            f'{count_value:,}  {percentage_value:.1f}%',
            va='center',
            fontsize=10,
            color=COLORS['text']
        )

def get_question_mean(question_column, data_frame):
    """Compute one question mean with missing values ignored."""
    return data_frame[question_column].dropna().mean()

def classify_question(mean, std_dev):
    """Classify a question using both central tendency and division level."""
    divided = std_dev >= 1.0
    if mean > 1.0:
        return 'Strong consensus — agree' if not divided else 'Majority agree — but divided'
    elif mean >= 0.5:
        return 'Moderate agreement' if not divided else 'Leaning agree — but divided'
    elif mean >= -0.5:
        return 'Genuinely neutral' if not divided else 'Polarised — two opposing camps'
    elif mean >= -1.0:
        return 'Moderate disagreement' if not divided else 'Leaning disagree — but divided'
    else:
        return 'Strong consensus — disagree' if not divided else 'Majority disagree — but divided'

def build_mean_table(question_ids, source_df, group_column, group_order):
    """Build a tidy mean-response table for a list of questions by subgroup."""
    table_rows = []
    for question_id in question_ids:
        question_column = f'q_{question_id}'
        for group_value in group_order:
            group_mask = source_df[group_column] == group_value
            group_series = source_df.loc[group_mask, question_column].dropna()
            table_rows.append({
                'group': group_value,
                'sample_size': int(group_series.shape[0]),
                'question_id': question_id,
                'question_text': QUESTION_TEXT[question_id],
                'mean_response': round(group_series.mean(), 2),
            })
    return pd.DataFrame(table_rows)


In [16]:
# Task 2: Political Orientation Overview
political_distribution_df = (
    df['resultLabel']
    .value_counts()
    .reindex(RESULT_LABEL_ORDER, fill_value=0)
    .rename_axis('result_label')
    .reset_index(name='count')
)
political_distribution_df['percentage'] = political_distribution_df['count'] / total_count * 100
save_table(political_distribution_df, '01_political_distribution.csv')

# Store this percentage for later narrative writing in findings.md.
economically_left_of_centre_pct = (df['economicScore'] < 0).mean() * 100

figure_01, axes_01 = plt.subplots(figsize=(10, 6), facecolor='white')
bar_colors = [LABEL_COLORS[label] for label in political_distribution_df['result_label']]
axes_01.barh(
    political_distribution_df['result_label'],
    political_distribution_df['count'],
    color=bar_colors
)
axes_01.invert_yaxis()
axes_01.set_title(f'Political Orientation of {total_count:,} Respondents', fontsize=16, pad=14)
axes_01.set_xlabel('Respondent count')
style_axes(axes_01)
add_count_and_pct_labels(
    axes_01,
    political_distribution_df['count'].tolist(),
    political_distribution_df['percentage'].tolist()
)
add_sample_caveat(figure_01)
save_chart(figure_01, '01_political_distribution.png')
plt.close(figure_01)


In [17]:
# Task 3: The Centrist Illusion
centrist_mask = df['resultLabel'] == 'Centrist'
centrist_df = df.loc[centrist_mask].copy()
pct_centrists_scoring_left = (centrist_df['economicScore'] < 0).mean() * 100

# Use the requested score buckets so the cross-tab matches the spec exactly.
score_bucket_conditions = [
    df['economicScore'] < -1.5,
    (df['economicScore'] >= -1.5) & (df['economicScore'] <= 1.5),
    df['economicScore'] > 1.5,
]
score_bucket_labels = ['Economically Left', 'Economically Centre', 'Economically Right']
df['economic_score_bucket'] = np.select(score_bucket_conditions, score_bucket_labels, default='Missing')

centrist_breakdown_df = pd.crosstab(
    df['economic_score_bucket'],
    df['resultLabel'],
    dropna=False
).reindex(index=score_bucket_labels, columns=RESULT_LABEL_ORDER, fill_value=0)
centrist_breakdown_df = centrist_breakdown_df.reset_index().rename(columns={'economic_score_bucket': 'score_bucket'})
save_table(centrist_breakdown_df, '02_centrist_breakdown.csv')

figure_02, axes_02 = plt.subplots(figsize=(9, 7), facecolor='white')
axes_02.scatter(
    df['economicScore'],
    df['socialScore'],
    s=8,
    alpha=0.15,
    color='#B0B0B0',
    edgecolor='none'
)
axes_02.scatter(
    centrist_df['economicScore'],
    centrist_df['socialScore'],
    s=12,
    alpha=0.4,
    color=COLORS['amber'],
    edgecolor='none'
)
axes_02.axvline(0, color=COLORS['muted'], linestyle='--', linewidth=1)
axes_02.axhline(0, color=COLORS['muted'], linestyle='--', linewidth=1)
axes_02.set_xlabel('Economic score')
axes_02.set_ylabel('Social score')
axes_02.set_title('Where do algorithm-classified Centrists actually sit on the compass?', fontsize=16, pad=14)
style_axes(axes_02)
axes_02.annotate(
    f'{pct_centrists_scoring_left:.1f}% of algorithm-classified Centrists sit left of x=0',
    xy=(0.02, 0.97),
    xycoords='axes fraction',
    ha='left',
    va='top',
    fontsize=11,
    color=COLORS['text'],
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['amber'])
)
add_sample_caveat(figure_02)
save_chart(figure_02, '02_centrist_illusion.png')
plt.close(figure_02)


In [18]:
# Task 4: The Gender Gap — Five Striking Questions
# Reference values from the spec:
# Q16 male=-0.23, female=+0.59, gap=0.82
# Q12 male=+0.19, female=+0.85, gap=0.67
# Q9  male=+0.32, female=-0.30, gap=0.62
# Q11 male=-0.11, female=-0.60, gap=0.48
# Q10 male=+0.84, female=+1.27, gap=0.43
gender_gap_rows = []
for question_column in question_cols:
    question_id = get_question_id(question_column)
    male_answers = df.loc[df['gender'] == 'male', question_column].dropna()
    female_answers = df.loc[df['gender'] == 'female', question_column].dropna()
    male_mean = male_answers.mean()
    female_mean = female_answers.mean()
    male_std_dev = male_answers.std()
    female_std_dev = female_answers.std()
    gap_value = female_mean - male_mean
    p_value = safe_ttest(male_answers, female_answers)
    gender_gap_rows.append({
        'question_id': question_id,
        'question_text': QUESTION_TEXT[question_id],
        'male_mean': round(male_mean, 2),
        'female_mean': round(female_mean, 2),
        'male_classification': classify_question(male_mean, male_std_dev),
        'female_classification': classify_question(female_mean, female_std_dev),
        'gap': round(gap_value, 2),
        'p_value': np.nan if pd.isna(p_value) else round(p_value, 6),
        'significant': False if pd.isna(p_value) else p_value < 0.05,
        'abs_gap': abs(gap_value),
    })

gender_gap_df = pd.DataFrame(gender_gap_rows).sort_values('abs_gap', ascending=False)
save_table(
    gender_gap_df[['question_id', 'question_text', 'male_mean', 'female_mean', 'male_classification', 'female_classification', 'gap', 'p_value', 'significant']],
    '03_gender_gap.csv'
)

# Keep these values for the final narrative summary.
q16_gender_row = gender_gap_df.loc[gender_gap_df['question_id'] == 16].iloc[0]
q16_male = q16_gender_row['male_mean']
q16_female = q16_gender_row['female_mean']
q16_gap = q16_gender_row['gap']

top_gender_gap_df = gender_gap_df.head(5).sort_values('abs_gap', ascending=True)
figure_03, axes_03 = plt.subplots(figsize=(11, 7), facecolor='white')
male_bar_values = -top_gender_gap_df['male_mean'].abs().to_numpy()
female_bar_values = top_gender_gap_df['female_mean'].abs().to_numpy()
y_positions = np.arange(len(top_gender_gap_df))
axes_03.barh(y_positions, male_bar_values, color=COLORS['primary'], label=f'Male (n={male_count:,})')
axes_03.barh(y_positions, female_bar_values, color=COLORS['amber'], label=f'Female (n={female_count:,})')
axes_03.axvline(0, color=COLORS['muted'], linewidth=1)
axes_03.set_yticks(y_positions)
axes_03.set_yticklabels([get_short_question_label(question_id) for question_id in top_gender_gap_df['question_id']])
axes_03.set_xlabel('Distance from neutral response')
axes_03.set_title('Biggest Gender Differences in Survey Responses', fontsize=16, pad=14)
style_axes(axes_03)
for row_position, row_data in enumerate(top_gender_gap_df.itertuples()):
    axes_03.text(male_bar_values[row_position] - 0.05, row_position, f"{row_data.male_mean:+.2f}", ha='right', va='center', fontsize=9)
    axes_03.text(female_bar_values[row_position] + 0.05, row_position, f"{row_data.female_mean:+.2f}", ha='left', va='center', fontsize=9)
axes_03.legend(loc='lower right', frameon=False)
figure_03.text(0.5, 0.02, 'Female sample is small (n=1,781). Interpret with caution.', ha='center', fontsize=10, color=COLORS['muted'])
add_sample_caveat(figure_03)
save_chart(figure_03, '03_gender_gap_questions.png')
plt.close(figure_03)


In [19]:
# Task 5: Most Agreed & Most Divisive Questions
# Reference values from the spec:
# Highest mean: Q17=+1.40, Q20=+1.15, Q19=+1.14
# Most polarised: Q9 std=1.48, Q16 std=1.44, Q12 std=1.43
# Most disagreed: Q1=-0.45
question_analysis_rows = []
for question_column in question_cols:
    question_id = get_question_id(question_column)
    mean_value = df[question_column].mean()
    std_value = df[question_column].std()
    question_analysis_rows.append({
        'question_id': question_id,
        'question_text': QUESTION_TEXT[question_id],
        'category': QUESTION_CATEGORY[question_id],
        'mean': round(mean_value, 2),
        'std_dev': round(std_value, 2),
        'classification': classify_question(mean_value, std_value),
    })

question_analysis_df = pd.DataFrame(question_analysis_rows)
question_analysis_df['classification'] = question_analysis_df.apply(
    lambda row: classify_question(row['mean'], row['std_dev']), axis=1
)
save_table(question_analysis_df, '04_question_analysis.csv')

# Keep the top consensus and divisive items for findings.md.
top_question_row = question_analysis_df.sort_values('mean', ascending=False).iloc[0]
top_q_id = int(top_question_row['question_id'])
top_q_text = top_question_row['question_text']
top_q_mean = float(top_question_row['mean'])
divisive_question_row = question_analysis_df.sort_values('std_dev', ascending=False).iloc[0]
divisive_q_id = int(divisive_question_row['question_id'])
divisive_q_text = divisive_question_row['question_text']
divisive_q_std = float(divisive_question_row['std_dev'])

mean_sorted_df = question_analysis_df.sort_values('mean', ascending=False).copy()
std_sorted_df = question_analysis_df.sort_values('std_dev', ascending=False).copy()
mean_bar_colors = []
for mean_value in mean_sorted_df['mean']:
    if mean_value > 0.5:
        mean_bar_colors.append(COLORS['green'])
    elif mean_value < -0.5:
        mean_bar_colors.append(COLORS['red'])
    else:
        mean_bar_colors.append(COLORS['amber'])

# Use a purple gradient so higher standard deviation looks visually stronger.
std_bar_colors = sns.light_palette(COLORS['purple'], n_colors=len(std_sorted_df), reverse=True)

figure_04, axes_04 = plt.subplots(1, 2, figsize=(16, 9), facecolor='white')
axes_04[0].barh(
    [get_short_question_label(question_id) for question_id in mean_sorted_df['question_id']],
    mean_sorted_df['mean'],
    color=mean_bar_colors
)
axes_04[0].invert_yaxis()
axes_04[0].set_xlabel('Mean response')
axes_04[0].set_title('Mean score')
style_axes(axes_04[0])

axes_04[1].barh(
    [get_short_question_label(question_id) for question_id in std_sorted_df['question_id']],
    std_sorted_df['std_dev'],
    color=std_bar_colors
)
axes_04[1].invert_yaxis()
axes_04[1].set_xlabel('Standard deviation (higher = more divided)')
axes_04[1].set_title('Standard deviation')
style_axes(axes_04[1])

figure_04.suptitle('What 17,259 Bangladeshis most agree and disagree with', fontsize=17, y=1.02)
add_sample_caveat(figure_04)
save_chart(figure_04, '04_question_consensus.png')
plt.close(figure_04)


In [20]:
# Task 6: Age vs Social Questions — The Generation Gap
age_social_df = build_mean_table(SOCIAL_QUESTION_IDS, df, 'age', AGE_ORDER)
save_table(age_social_df, '05_age_social.csv')

age_social_pivot = age_social_df.pivot(index='group', columns='question_id', values='mean_response').reindex(AGE_ORDER)
age_group_counts = df['age'].value_counts().reindex(AGE_ORDER, fill_value=0)
age_row_labels = []
for age_group in AGE_ORDER:
    row_label = f"{age_group} (n={age_group_counts[age_group]:,})"
    if age_group in ['45-54', '55+']:
        row_label = row_label + ' *'
    age_row_labels.append(row_label)

# Store the specific age comparison requested in findings.md.
q20_young = float(age_social_pivot.loc['18-24', 20])
q20_mid = float(age_social_pivot.loc['35-44', 20])

figure_05, axes_05 = plt.subplots(figsize=(12, 5), facecolor='white')
sns.heatmap(
    age_social_pivot,
    annot=True,
    fmt='.1f',
    cmap='RdBu_r',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean response'},
    ax=axes_05
)
axes_05.set_yticklabels(age_row_labels, rotation=0)
axes_05.set_xticklabels([get_short_question_label(question_id) for question_id in age_social_pivot.columns], rotation=30, ha='right')
axes_05.set_title('Social Question Responses by Age Group', fontsize=16, pad=14)
axes_05.set_xlabel('Social questions')
axes_05.set_ylabel('Age group')
style_axes(axes_05)
add_sample_caveat(figure_05)
save_chart(figure_05, '05_age_social_heatmap.png')
plt.close(figure_05)


In [21]:
# Task 7: Education vs Tolerance — Does Higher Education Mean More Open?
education_tolerance_df = build_mean_table(TOLERANCE_QUESTION_IDS, df, 'education', EDUCATION_ORDER)
save_table(education_tolerance_df, '06_education_tolerance.csv')

education_plot_df = education_tolerance_df.copy()
education_plot_df['education_label'] = education_plot_df['group'].map(EDUCATION_LABELS)

# Store the key education contrast requested for findings.md.
q21_low_edu = float(education_tolerance_df.loc[(education_tolerance_df['group'] == 'below_ssc') & (education_tolerance_df['question_id'] == 21), 'mean_response'].iloc[0])
q21_high_edu = float(education_tolerance_df.loc[(education_tolerance_df['group'] == 'bachelor_masters') & (education_tolerance_df['question_id'] == 21), 'mean_response'].iloc[0])

question_positions = np.arange(len(TOLERANCE_QUESTION_IDS))
bar_width = 0.18
figure_06, axes_06 = plt.subplots(figsize=(14, 6), facecolor='white')
for group_index, education_key in enumerate(EDUCATION_ORDER):
    group_values = education_plot_df.loc[education_plot_df['group'] == education_key, 'mean_response'].to_numpy()
    axes_06.bar(
        question_positions + (group_index - 1.5) * bar_width,
        group_values,
        width=bar_width,
        color=CAT_COLORS[group_index],
        label=EDUCATION_LABELS[education_key]
    )
axes_06.set_xticks(question_positions)
axes_06.set_xticklabels([get_short_question_label(question_id) for question_id in TOLERANCE_QUESTION_IDS], rotation=25, ha='right')
axes_06.set_ylabel('Mean response')
axes_06.set_title('Responses to Tolerance Questions by Education Level', fontsize=16, pad=14)
axes_06.legend(frameon=False, ncol=2)
style_axes(axes_06)
add_sample_caveat(figure_06)
save_chart(figure_06, '06_education_tolerance.png')
plt.close(figure_06)


In [22]:
# Task 8: The Five Most Divisive Questions — Full Response Distribution
top_divisive_question_ids = question_analysis_df.sort_values('std_dev', ascending=False).head(5)['question_id'].tolist()
divisive_distribution_rows = []
for question_id in top_divisive_question_ids:
    question_column = f'q_{question_id}'
    response_counts = df[question_column].value_counts().sort_index().reindex([-2, -1, 0, 1, 2], fill_value=0)
    response_percentages = response_counts / response_counts.sum() * 100
    for response_value in [-2, -1, 0, 1, 2]:
        divisive_distribution_rows.append({
            'question_id': question_id,
            'question_text': QUESTION_TEXT[question_id],
            'response_value': response_value,
            'count': int(response_counts.loc[response_value]),
            'percentage': round(response_percentages.loc[response_value], 2),
        })

divisive_distribution_df = pd.DataFrame(divisive_distribution_rows)
save_table(divisive_distribution_df, '07_divisive_distributions.csv')

distribution_colors = ['#C0392B', '#E74C3C', '#95A5A6', '#27AE60', '#1E8449']
response_value_labels = ['Strongly disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly agree']
figure_07, axes_07 = plt.subplots(2, 3, figsize=(18, 8), facecolor='white')
axes_07 = axes_07.flatten()
for axes_index, question_id in enumerate(top_divisive_question_ids):
    question_column = f'q_{question_id}'
    response_counts = df[question_column].value_counts().sort_index().reindex([-2, -1, 0, 1, 2], fill_value=0)
    axes_07[axes_index].bar(response_value_labels, response_counts.values, color=distribution_colors)
    axes_07[axes_index].set_title(get_short_question_label(question_id), fontsize=11)
    axes_07[axes_index].tick_params(axis='x', rotation=45)
    style_axes(axes_07[axes_index])
for empty_axes in axes_07[len(top_divisive_question_ids):]:
    empty_axes.axis('off')
figure_07.suptitle('Response Distribution for the 5 Most Polarising Questions', fontsize=17, y=1.02)
add_sample_caveat(figure_07)
save_chart(figure_07, '07_divisive_distributions.png')
plt.close(figure_07)


In [23]:
# Task 9: Probashi vs Bangladesh — What Living Abroad Changes
probashi_rows = []
for question_column in question_cols:
    question_id = get_question_id(question_column)
    home_answers = df.loc[df['probashi'] == 'no', question_column].dropna()
    abroad_answers = df.loc[df['probashi'] == 'yes', question_column].dropna()
    home_std_dev = home_answers.std()
    abroad_std_dev = abroad_answers.std()
    p_value = safe_ttest(home_answers, abroad_answers)
    probashi_rows.append({
        'question_id': question_id,
        'question_text': QUESTION_TEXT[question_id],
        'home_mean': round(home_answers.mean(), 2),
        'abroad_mean': round(abroad_answers.mean(), 2),
        'home_classification': classify_question(home_answers.mean(), home_std_dev),
        'abroad_classification': classify_question(abroad_answers.mean(), abroad_std_dev),
        'gap': round(abroad_answers.mean() - home_answers.mean(), 2),
        'p_value': np.nan if pd.isna(p_value) else round(p_value, 6),
        'significant': False if pd.isna(p_value) else p_value < 0.05,
    })

probashi_comparison_df = pd.DataFrame(probashi_rows)
save_table(
    probashi_comparison_df[['question_id', 'question_text', 'home_mean', 'abroad_mean', 'home_classification', 'abroad_classification', 'gap', 'p_value']],
    '08_probashi_comparison.csv'
)

# Keep the Q20 values for the final findings file.
q20_probashi_row = probashi_comparison_df.loc[probashi_comparison_df['question_id'] == 20].iloc[0]
q20_home = q20_probashi_row['home_mean']
q20_abroad = q20_probashi_row['abroad_mean']
q20_diff = q20_probashi_row['gap']

probashi_plot_df = probashi_comparison_df.copy()
probashi_plot_df['abs_gap'] = probashi_plot_df['gap'].abs()
probashi_plot_df = probashi_plot_df.sort_values('abs_gap', ascending=False)
figure_08, axes_08 = plt.subplots(figsize=(12, 10), facecolor='white')
y_positions = np.arange(len(probashi_plot_df))
axes_08.hlines(y_positions, probashi_plot_df['home_mean'], probashi_plot_df['abroad_mean'], color=COLORS['muted'], linewidth=1.5)
axes_08.scatter(probashi_plot_df['home_mean'], y_positions, color=COLORS['primary'], s=45, label='Bangladesh')
axes_08.scatter(probashi_plot_df['abroad_mean'], y_positions, color=COLORS['green'], s=45, label='Probashi')
for row_position, row_data in enumerate(probashi_plot_df.itertuples()):
    if row_data.significant:
        axes_08.text(max(row_data.home_mean, row_data.abroad_mean) + 0.08, row_position, '*', fontsize=14, va='center')
axes_08.set_yticks(y_positions)
axes_08.set_yticklabels([get_short_question_label(question_id) for question_id in probashi_plot_df['question_id']])
axes_08.invert_yaxis()
axes_08.set_xlabel('Mean response')
axes_08.set_title('How Probashi Bangladeshis differ from those at home', fontsize=16, pad=14)
axes_08.legend(frameon=False)
style_axes(axes_08)
add_sample_caveat(figure_08)
save_chart(figure_08, '08_probashi_comparison.png')
plt.close(figure_08)


In [24]:
# Task 10: Occupation — Businesspeople vs Everyone Else
occupation_score_rows = []
for occupation_value in OCCUPATION_ORDER:
    occupation_mask = df['occupation'] == occupation_value
    occupation_df = df.loc[occupation_mask]
    row_data = {
        'occupation': occupation_value,
        'sample_size': int(len(occupation_df)),
        'economicScore_mean': round(occupation_df['economicScore'].mean(), 2),
        'socialScore_mean': round(occupation_df['socialScore'].mean(), 2),
    }
    for question_id in ECONOMIC_OCCUPATION_QUESTION_IDS:
        row_data[f'q_{question_id}_mean'] = round(occupation_df[f'q_{question_id}'].mean(), 2)
    occupation_score_rows.append(row_data)

occupation_scores_df = pd.DataFrame(occupation_score_rows)
save_table(occupation_scores_df, '09_occupation_scores.csv')

occupation_score_plot_df = occupation_scores_df.sort_values('economicScore_mean', ascending=True)
left_bar_colors = [COLORS['primary'] if score_value < 0 else COLORS['amber'] for score_value in occupation_score_plot_df['economicScore_mean']]

# Build a tidy frame for the grouped question chart.
occupation_question_rows = []
for occupation_value in OCCUPATION_ORDER:
    for question_id in [3, 7, 8]:
        occupation_question_rows.append({
            'occupation': occupation_value,
            'occupation_label': OCCUPATION_LABELS[occupation_value],
            'question_id': question_id,
            'mean_response': get_question_mean(f'q_{question_id}', df.loc[df['occupation'] == occupation_value])
        })
occupation_question_df = pd.DataFrame(occupation_question_rows)

figure_09, axes_09 = plt.subplots(1, 2, figsize=(16, 7), facecolor='white')
axes_09[0].barh(
    [OCCUPATION_LABELS[occupation_value] for occupation_value in occupation_score_plot_df['occupation']],
    occupation_score_plot_df['economicScore_mean'],
    color=left_bar_colors
)
axes_09[0].set_xlabel('Mean economic score')
axes_09[0].set_title('Economic score by occupation')
style_axes(axes_09[0])

occupation_question_ids_for_chart = [3, 7, 8]
question_positions = np.arange(len(occupation_question_ids_for_chart))
bar_width = 0.14
for occupation_index, occupation_value in enumerate(OCCUPATION_ORDER):
    group_values = []
    for question_id in occupation_question_ids_for_chart:
        value = occupation_question_df.loc[
            (occupation_question_df['occupation'] == occupation_value) & (occupation_question_df['question_id'] == question_id),
            'mean_response'
        ].iloc[0]
        group_values.append(value)
    axes_09[1].bar(
        question_positions + (occupation_index - 2) * bar_width,
        group_values,
        width=bar_width,
        color=CAT_COLORS[occupation_index],
        label=OCCUPATION_LABELS[occupation_value]
    )
axes_09[1].set_xticks(question_positions)
axes_09[1].set_xticklabels([get_short_question_label(question_id) for question_id in occupation_question_ids_for_chart], rotation=20, ha='right')
axes_09[1].set_ylabel('Mean response')
axes_09[1].set_title('Q3, Q7 and Q8 by occupation')
axes_09[1].legend(frameon=False, fontsize=9)
style_axes(axes_09[1])

figure_09.suptitle('Economic Views by Occupation', fontsize=17, y=1.02)
add_sample_caveat(figure_09)
save_chart(figure_09, '09_occupation_economics.png')
plt.close(figure_09)


In [25]:
# Task 11: Growth Over Time — The Viral Story
daily_growth_df = (
    df.groupby('response_date')
    .size()
    .rename('daily_count')
    .reset_index()
    .sort_values('response_date')
)
daily_growth_df['cumulative_total'] = daily_growth_df['daily_count'].cumsum()
daily_growth_df['date'] = pd.to_datetime(daily_growth_df['response_date'])
save_table(daily_growth_df[['date', 'daily_count', 'cumulative_total']], '10_daily_growth.csv')

peak_growth_row = daily_growth_df.loc[daily_growth_df['daily_count'].idxmax()]
peak_date = format_date_label(peak_growth_row['date'])
peak_count = int(peak_growth_row['daily_count'])
total_days = int(len(daily_growth_df))
avg_per_day = daily_growth_df['daily_count'].mean()

figure_10, axes_10 = plt.subplots(2, 1, figsize=(13, 9), facecolor='white', sharex=True)
date_labels = [format_date_label(date_value) for date_value in daily_growth_df['date']]
daily_bar_colors = [COLORS['primary']] * len(daily_growth_df)
daily_bar_colors[int(daily_growth_df['daily_count'].idxmax())] = COLORS['amber']
axes_10[0].bar(date_labels, daily_growth_df['daily_count'], color=daily_bar_colors)
axes_10[0].set_ylabel('Daily responses')
axes_10[0].set_title(f'How {total_count:,} Bangladeshis discovered Bam-Dan in {total_days} days', fontsize=16, pad=14)
style_axes(axes_10[0])
peak_label_x = daily_growth_df.index[daily_growth_df['daily_count'].idxmax()]
axes_10[0].annotate(
    f'Peak: {peak_count:,} responses',
    xy=(peak_label_x, peak_count),
    xytext=(peak_label_x, peak_count + daily_growth_df['daily_count'].max() * 0.08),
    ha='center',
    arrowprops=dict(arrowstyle='->', color=COLORS['muted']),
    fontsize=10
)

axes_10[1].plot(date_labels, daily_growth_df['cumulative_total'], color=COLORS['primary'], linewidth=2)
axes_10[1].fill_between(date_labels, daily_growth_df['cumulative_total'], color=COLORS['primary'], alpha=0.2)
axes_10[1].set_ylabel('Cumulative total')
axes_10[1].annotate(
    f"{int(daily_growth_df['cumulative_total'].iloc[-1]):,}",
    xy=(len(date_labels) - 1, daily_growth_df['cumulative_total'].iloc[-1]),
    xytext=(len(date_labels) - 1, daily_growth_df['cumulative_total'].iloc[-1] + daily_growth_df['daily_count'].max()),
    ha='center',
    arrowprops=dict(arrowstyle='->', color=COLORS['muted']),
    fontsize=10
)
style_axes(axes_10[1])
plt.setp(axes_10[1].get_xticklabels(), rotation=45, ha='right')
add_sample_caveat(figure_10)
save_chart(figure_10, '10_growth_over_time.png')
plt.close(figure_10)


In [26]:
# Task 12: Auto-generate findings.md
today_label = pd.Timestamp.today().strftime('%Y-%m-%d')
sample_start_label = format_date_label(sample_start_date)
sample_end_label = format_date_label(sample_end_date)

# Use computed values only so the findings file always matches the current data.
findings_text = f"""# Bam-Dan Survey Key Findings
Generated: {today_label}
Sample: {total_count} responses | {sample_start_label}-{sample_end_label} {sample_year} | bam-dan.com

## IMPORTANT: Sample Caveat
Self-selected online sample. {male_pct:.1f}% male, {student_pct:.1f}% students.
These findings describe Bangladesh's politically engaged online youth,
not the general population.

## 1. The Centrist Illusion
{pct_centrists_scoring_left:.1f}% of algorithm-classified Centrists have a
negative (left-of-centre) economic score.

## 2. Biggest Gender Gap
Q16 (Equal property rights for men and women):
  Male mean = {q16_male:.2f}, Female mean = {q16_female:.2f}
  Gap = {q16_gap:.2f} — largest gender difference in the dataset.

## 3. Most Agreed Question
Q{top_q_id} - \"{top_q_text}\"
  Mean response: {top_q_mean:.2f} (near-universal agreement)

## 4. Most Divisive Question
Q{divisive_q_id} - \"{divisive_q_text}\"
  Standard deviation: {divisive_q_std:.2f} — bimodal distribution,
  two strong opposing camps.

## 5. Generation Surprise
Q20 (Strictly ban anything hurting religious sentiments):
  18-24 age group mean = {q20_young:.2f}
  35-44 age group mean = {q20_mid:.2f}
  Younger respondents are stricter on religious speech.

## 6. Education and Minority Rights
Q21 (Special rights for indigenous/ethnic minorities):
  Below SSC mean = {q21_low_edu:.2f}
  Bachelor/Masters mean = {q21_high_edu:.2f}
  Education is the strongest predictor of support for minority rights.

## 7. Probashi Difference
Q20 (Strictly ban anything hurting religious sentiments):
  Bangladesh respondents mean = {q20_home:.2f}
  Probashi (abroad) mean = {q20_abroad:.2f}
  Difference = {q20_diff:.2f}

## 8. Growth Story
Peak day: {peak_date} ({peak_count} responses in one day)
Total responses: {total_count} over {total_days} days
Average per day: {avg_per_day:.0f}
"""

with open('private_analysis/findings.md', 'w', encoding='utf-8') as findings_file:
    findings_file.write(findings_text)

print('Notebook code is ready. Run all cells to generate outputs.')


Notebook code is ready. Run all cells to generate outputs.
